# S3 J6 - Context Engineering

# Objectifs

## Comprendre et maîtriser :

- Context Engineering (concept central des agents modernes)
- Construction du contexte envoyé aux LLM
- Sélection dynamique des informations
- Compression / résumé de contexte
- Windowing (fenêtre de contexte)
- Memory vs Context vs State
- Stratégies de réduction des tokens
- Qualité des réponses LLM
- Optimisation coût / performance

# Context Engineering

Objectif :

Comprendre comment construire
le contexte optimal envoyé à un LLM.

# Définition

Context Engineering =

"Sélectionner, structurer et optimiser
les informations envoyées au modèle."

# Pourquoi c'est critique ?

Un LLM ne "se souvient pas".

Il reçoit un contexte à chaque appel.
```
User
  ↓
Agent
  ↓
Conversation Messages
  ↓
Context Builder
  ↓
LLM
  ↓
Réponse
```

# Problème réel

Tu ne peux pas envoyer :

- toute la conversation
- toute la base de données
- toute la mémoire

# Limite des tokens

Chaque modèle a une fenêtre :

ex :

- 8k tokens
- 32k tokens
- 128k tokens

# Objectif du Context Engineering

Maximiser :

- pertinence
- précision
- cohérence

Minimiser :

- tokens
- bruit

# Architecture globale
```
Memory (PostgreSQL)
        ↓
State (Redis)
        ↓
Context Builder
        ↓
LLM
```
# Exemple concret

Utilisateur :

"Prépare un voyage à Rome"

# Mauvais contexte

- toute l'historique
- toutes les préférences
- tous les messages

# Bon contexte

- destination = Rome
- budget = 1200€
- dates = juin
- préférences = hôtel centre-ville

Chaque agent reçoit un contexte différent.
```
Planner
    ↓
Seulement le plan.
```
```
Researcher
    ↓
Documents
    ↓
Recherche
```
```
Writer
    ↓
Résumé
    ↓
Résultats
```

# Construction du contexte
### Les 7 sources de contexte

C'est ici que je modifierais complètement le notebook.
```
             Context Builder

                    │

 ┌──────────────────┼────────────────────┐

 ▼                  ▼                    ▼

Messages        Shared State       Long Memory

 ▼                  ▼                    ▼

Tool Results     RAG Docs       Runtime Metadata

                    ▼

             System Prompt
```
Étapes :

1. Identifier l'intention
2. Charger le state
3. Charger la mémoire utile
4. Filtrer
5. Structurer

### Source 1

Conversation Messages

Exemple :
```
messages = [
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
]
```
Expliquer :

Pourquoi il faut les conserver.

Pourquoi il ne faut pas tout conserver.

Sélection des messages

Exemple :

```
def select_recent_messages(
    messages,
    n=10
):
    return messages[-n:]
```
Résumé de conversation

Au lieu de :
```
100 messages
↓
Résumé
↓
5 messages
```
### Source 2
State

Redis
```
state = {

    "destination": "Rome",

    "budget": 1200,

    "step": "hotel"
}
```
### Source 3
Memory

PostgreSQL

Préférences utilisateur.
### Source 4
Tool Results

Très souvent oubliés.

Exemple :
```
tool_results = {

    "weather":

    "26°C",

    "flight_search":

    [...]
}
```
### Source 5
Documents RAG

On ajoute seulement les documents utiles.
### Source 6
Runtime Metadata

Exemple :
```
metadata = {

    "date": "...",

    "timezone": "...",

    "language": "fr",

    "remaining_budget": 0.35
}
```
### Source 7
System Prompt

Toujours présent.

Toujours envoyé.

Version complète

In [ ]:
class ContextBuilder:

    def build(

        self,

        messages,

        state,

        memory,

        tool_results,

        documents,

        metadata,

        system_prompt

    ):
        ...

Pipeline
```
Conversation
↓
Select Messages
↓
Retrieve Memory
↓
Retrieve RAG
↓
Retrieve Tool Results
↓
Merge
↓
Compress
↓
LLM
```

# Filtrage (Core Concept)

In [ ]:
def is_relevant(item, query):

    return query.lower() in item.lower()

# RAG vs Context Engineering

RAG :

→ récupère des documents

Context Engineering :

→ construit le prompt final

# Compression du contexte

In [ ]:
def compress(texts):

    return " | ".join(texts[:3])

# Résumé automatique

In [ ]:
def summarize(messages):

    return "Résumé de la conversation"

# Context Window Strategy

Stratégies :

- FIFO (First In First Out)
- Importance scoring
- Summarization
- Hybrid

In [ ]:
def select_context(messages, limit=5):

    return messages[-limit:]

# Importance scoring

In [ ]:
def score(item):

    keywords = ["budget", "objectif", "projet"]

    return sum(
        1 for k in keywords
        if k in item.lower()
    )

# Pipeline complet

In [ ]:
def context_pipeline(state, memory, messages):

    context = build_context(state, memory)

    context["messages"] = select_context(messages)

    return context

# Exemple complet

In [ ]:
state = {
    "task": "travel",
    "budget": 1200
}

memory = {
    "preferences": ["calme", "centre"]
}

messages = [
    "bonjour",
    "je veux voyager",
    "Rome"
]

context_pipeline(state, memory, messages)

# Observabilité du contexte

In [ ]:
def context_size(context):

    return len(str(context))

# Coût LLM

Plus le contexte est grand :
```
→ plus cher
→ plus lent
→ parfois moins précis
```

# Anti-patterns

❌ Envoyer toute la mémoire

❌ Envoyer toute la conversation

❌ Pas de filtrage

❌ Pas de compression

# Architecture cible
```
                 User

                  │

         Conversation Messages

                  │

                  ▼

          Conversation Store

                  │

       ┌──────────┼───────────┐

       ▼          ▼           ▼

Redis State   PostgreSQL   Tool Cache

                    │

             RAG Retrieval

                    │

                    ▼

           Context Builder

                    │

                    ▼

         OpenAI Responses API

                    │

                    ▼

               Tool Calling

                    │

                    ▼

               MCP Servers
```

# Mini Projet

Construire un Context Builder pour :

- agent de voyage
- agent support
- agent assistant personnel
  
# Exercice 1

Créer un filtre de mémoire pertinente

# Exercice 2

Implémenter un scoring de contexte

# Exercice 3

Ajouter compression du contexte

# Exercice 4

Mesurer le coût token estimé

# Exercice 5

Comparer :

- contexte brut
- contexte optimisé

# Notes d'architecte

Le Context Engineering est :

le véritable cœur des agents modernes.

Sans lui :

- coûts explosent
- qualité chute
- comportements instables

Avec lui :

- contrôle
- performance
- scalabilité

# Conclusion pédagogique

Tu es maintenant sur le noyau dur des systèmes IA modernes :
```
Agents
+ MCP
+ State (Redis)
+ Memory (PostgreSQL)
+ Context Engineering
= AI Systems réels
```